# Auto Price Prediction Application
# Jupyter Notebook #4: Model Refinement and Final Training

**Developer:** Dominique Zuniga   
**Date:** July 2, 2025  
**Purpose:** This is the final notebook in the main pipeline. Its purpose is to take the best hyperparameters found by the Optuna study, build a refined model with those settings, train it on the complete training dataset, and perform a final evaluation on the hold-out test set to determine the model's real-world performance.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

try:
    processed_data_path = '../data/processed' # Load artifacts/tensors
    X_train_tensor = torch.load(f'{processed_data_path}/X_train_tensor.pt')
    y_train_tensor = torch.load(f'{processed_data_path}/y_train_tensor.pt')
    X_test_tensor = torch.load(f'{processed_data_path}/X_test_tensor.pt')
    y_test_tensor = torch.load(f'{processed_data_path}/y_test_tensor.pt')
    input_features = X_train_tensor.shape[1]

    preprocessor_path = '../saved_models/preprocessor.joblib' # Load the preprocessor
    preprocessor = joblib.load(preprocessor_path)

    print(f"\n✅ Data tensors and preprocessor loaded successfully.")
    print(f"Input features for model: {input_features}")

except FileNotFoundError as e:
    print(f"\n❌ ERROR: Could not load artifacts. Details: {e}")
    print("Please ensure you have run the '02_feature_engineering...' notebook first.")

try:
    raw_file_path = '../data/raw/used_cars.csv'
    df_raw = pd.read_csv(raw_file_path) # We need the raw data to create a fresh answer sheet; perhaps redundant?

    # Re-apply the same cleaning from notebook #2 to ensure alignment
    df_raw['price'] = pd.to_numeric(df_raw['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
    df_raw.dropna(subset=['price'], inplace=True)

    y_full = df_raw['price'] # Dataset will be split therefore we need the whole dataset
    X_full = df_raw.drop(columns=['price'])

    # Here the data is split, the testing set is comprised of 20% of data, the random split must match prior declaration (42 is a popular convention)
    _, X_test_original, _, y_test_original = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

    print("\n✅ Original test data reconstructed for error analysis.")

except FileNotFoundError as e:
    print(f"❌ Could not load raw CSV for analysis. Details: {e}")

In [ ]:
# Cell 2: Load Baseline Model and Generate Predictions

# DEFINE THE MODEL ARCHITECTURE
# This MUST EXACTLY MATCH the architecture of the model you are loading.
class PricePredictor(nn.Module):
    def __init__(self, num_input_features, dropout_rate=0.4):
        super(PricePredictor, self).__init__()
        self.layer_1 = nn.Linear(num_input_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=dropout_rate)
        self.layer_2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=dropout_rate)
        self.layer_3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(p=dropout_rate)
        self.output_layer = nn.Linear(64, 1)
    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu1(x)
        x = self.bn1(x)
        x = self.dropout1(x)
        x = self.layer_2(x)
        x = self.relu2(x)
        x = self.bn2(x)
        x = self.dropout2(x)
        x = self.layer_3(x)
        x = self.relu3(x)
        x = self.bn3(x)
        x = self.dropout3(x)
        return self.output_layer(x)

baseline_model = PricePredictor(input_features) # Load model
model_load_path = '../saved_models/price_predictor_refined_v1.pth' 

try:
    baseline_model.load_state_dict(torch.load(model_load_path)) # State_dict holds information
    baseline_model.eval() # Set to evaluation mode
    print(f"✅ Baseline model loaded from '{model_load_path}'")
except Exception as e:
    print(f"❌ ERROR loading baseline model. Details: {e}")

with torch.no_grad(): # Running with no gradient tells PyTorch not to do learning calculations
    y_pred_log_tensor = baseline_model(X_test_tensor) # The model has never seen this test data


y_pred_actual_dollars = np.expm1(y_pred_log_tensor.numpy()) # This log conversion leaves us with plain dollar values
y_test_actual_dollars = np.expm1(y_test_tensor.numpy())

print("\nPredictions generated and converted back to dollar amounts.")



In [ ]:
# Cell 3: Deep Error Analysis - Find the Worst Predictions

if 'X_test_original' in locals():# Create an Analysis DataFrame
    analysis_df = X_test_original.copy() # Original test data is copied and named analysis_df
    analysis_df['actual_price'] = y_test_original.values # Adds new column 'actual_price' and fills it with answer sheet 'y_test_original'
    analysis_df['predicted_price'] = y_pred_actual_dollars # Adds new column 'predicted_price' and populates with model predictions
    analysis_df['error'] = analysis_df['predicted_price'] - analysis_df['actual_price'] # Creates 'error' column and populates with difference
    analysis_df['abs_error'] = np.abs(analysis_df['error']) # Gives us the error amount in plain $$$
    analysis_df_sorted = analysis_df.sort_values(by='abs_error', ascending=False) # Sort by the absolute error to see the biggest mistakes

    print("--- Top 20 Worst Predictions (Largest Errors) ---") # This is useful for understanding where the model is weakest
    display_cols = ['brand', 'model', 'model_year', 'milage', 'actual_price', 'predicted_price', 'error']
    display(analysis_df_sorted[display_cols].head(20).style.format({
        'actual_price': '${:,.2f}',
        'predicted_price': '${:,.2f}',
        'error': '${:,.2f}'
    }))
else:
    print("Could not create analysis DataFrame. 'X_test_original' not found.")



In [ ]:
# Cell 4: Visualize the Errors
# This cell plots the errors (residuals) to help you understand 
# the overall performance and identify any systematic bias.

# Predicted Price vs. Actual Price
plt.figure(figsize=(10, 10))
plt.scatter(analysis_df['actual_price'], analysis_df['predicted_price'], alpha=0.3)
plt.plot([analysis_df['actual_price'].min(), analysis_df['actual_price'].max()],
         [analysis_df['actual_price'].min(), analysis_df['actual_price'].max()],
         'r--', lw=2, label='Ideal Fit')
plt.title('Predicted vs. Actual Prices')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Cell 5: Hyperparameter Tuning with Optuna
# This cell uses the Optuna library to automatically search for the best 
# combination of hyperparameters (learning rate, dropout, etc.) to improve your model.

class CarPriceDataset(Dataset):
    def __init__(self, features, labels): # This is a custom Dataset class for PyTorch
        self.features = features          # It is necessary for the PyTorch DataLoader to work properlt
        self.labels = labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Define the objective function for Optuna to optimize
def objective(trial):
    # --- Suggest hyperparameters ---
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True) # Suggests a learning rate from 0.0001 to 0.01 on a log scale
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)  # Suggests a dropout rate uniformly between 0.1 and 0.5
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128]) # Suggests a batch size from the list of choices [32, 64, 128]
    
    # --- Define Model and Dataloaders ---
    model = PricePredictor(input_features, dropout_rate=dropout_rate) 
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_dataset = CarPriceDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # --- Training Loop (for a fixed number of epochs) ---
    epochs_for_tuning = 30 # Use fewer epochs for faster tuning
    for epoch in range(epochs_for_tuning):
        model.train()
        for batch_features, batch_labels in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
            
    # --- Evaluate and Return Validation Metric ---
    model.eval()
    with torch.no_grad():
        y_pred_tensor = model(X_test_tensor)
        validation_loss = criterion(y_pred_tensor, y_test_tensor)
        
    return validation_loss.item()

# --- Run the Optuna Study ---
print("\n--- Starting Optuna Hyperparameter Study ---")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15) # n_trials is how many combinations to test

print("\n--- Optuna Study Complete ---")
print("Best trial:")
best_trial = study.best_trial
print(f"  Value (Validation MSE): {best_trial.value}")
print("  Best Parameters: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")


In [ ]:
# Cell 6: Train, Evaluate, and Save the Final Refined Model
# This final cell takes the best parameters found by Optuna, trains a new model 
# from scratch with them for a full number of epochs, evaluates it, and saves 
# it with a new version name.

# --- Step 1: Get Best Hyperparameters from Study ---
best_params = study.best_params # Retrieves the dictionary of the best hyperparameters found by the Optuna study
refined_model = PricePredictor( # Creates a new, refined model instance using features and dropout rate from Optuna
    num_input_features=input_features, 
    dropout_rate=best_params['dropout_rate']
)
refined_optimizer = optim.Adam(refined_model.parameters(), lr=best_params['lr']) # Adam uses best learning raate
refined_criterion = nn.MSELoss() # Define loss function; MSE in this case

final_train_dataset = CarPriceDataset(X_train_tensor, y_train_tensor) # Wraps the final training data into a custom PyTorch Dataset object
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params['batch_size'], shuffle=True)  # Creates the final data loader using the best batch size

# --- Step 2: Full Training with Best Parameters ---
epochs_final = 300 # Train for more epochs now
print(f"\n--- Training Final Refined Model for {epochs_final} epochs ---")

for epoch in range(epochs_final):
    refined_model.train()
    for batch_features, batch_labels in final_train_loader:  # Starts the inner loop to iterate through all batches of data
        refined_optimizer.zero_grad()  # Clears the gradients from the previous batch to prevent them from accumulating
        outputs = refined_model(batch_features) # Iterates through model
        loss = refined_criterion(outputs, batch_labels) # Calculates loss
        loss.backward() # Calculates each parameter's share of the total loss
        refined_optimizer.step() # Adjusts weights 
    if (epoch + 1) % 50 == 0: # Prints every 50th epoch
        print(f"Epoch [{epoch+1}/{epochs_final}], Loss: {loss.item():.4f}")

# --- Step 3: Final Evaluation ---
refined_model.eval()
with torch.no_grad():
    y_pred_final_tensor = refined_model(X_test_tensor)
    
y_pred_final_np = y_pred_final_tensor.numpy()
y_test_np = y_test_tensor.numpy()

mae_final = mean_absolute_error(np.expm1(y_test_np), np.expm1(y_pred_final_np))
r2_final = r2_score(y_test_np, y_pred_final_np)

print("\n--- Refined Model Evaluation on Test Set ---")
print(f"Mean Absolute Error (MAE): ${mae_final:,.2f}")
print(f"R-squared (R2 Score): {r2_final:.4f}")

# --- Step 4: Save the Refined Model ---
refined_model_save_path = '../saved_models/price_predictor_refined_v1.pth'
torch.save(refined_model.state_dict(), refined_model_save_path)
print(f"\n✅ Refined model saved to '{refined_model_save_path}'")
